1. EDA baseline & data reconciliation
Memastikan dataset yang dianalisis Python sesuai dengan tabel PostgreSQL. Periksa grain, distribusi status, nilai yang tidak valid, dan konsistensi KPI.


In [1]:
import pandas as pd
from pathlib import Path

In [2]:
data_path = Path("../../Data/Processed/dataco_cleaned.csv")

if not data_path.exists():
    raise FileNotFoundError(
        f"Processed dataset not found{data_path}\n"
    )

df = pd.read_csv(
    data_path,
    low_memory=False,
    parse_dates= [
        "order date (DateOrders)",
        "shipping date (DateOrders)"
    ],
)

print(f"loaded file: {data_path}")
print(f"rows:{len(df):,}")
print(f"columns: {df.shape[1]}")


loaded file: ../../Data/Processed/dataco_cleaned.csv
rows:180,519
columns: 64


In [3]:

grain_check = pd.DataFrame({
    "metric": [
        "total_rows",
        "unique_order_items",
        "unique_orders",
        "duplicate_order_item_ids",
        "duplicate_full_rows"
    ],
    "value": [
        len(df),
        df["Order Item Id"].nunique(),
        df["Order Id"].nunique(),
        df["Order Item Id"].duplicated().sum(),
        df.duplicated().sum()
    ]
})

display(grain_check)


,metric,value
0,total_rows,180519
1,unique_order_items,180519
2,unique_orders,65752
3,duplicate_order_item_ids,0
4,duplicate_full_rows,0


Ada sekitar rata-rata 2.75 item per order karena 1 order bisa memiliki beberapa item

In [6]:
delivery_summary = (
    df["delivery_timing"].value_counts(dropna=False).rename_axis("delivery_timing").reset_index(name="item_count")
)

delivery_summary["percentage"] = (
    delivery_summary["item_count"]/ len(df)*100
)

display(delivery_summary.round(2))

,delivery_timing,item_count,percentage
0,Delayed,98977,54.83
1,Not delayed,73788,40.88
2,Canceled,7754,4.30


1. 54,83% item terlambat.
2. 40.88% item tidak terlambat
3. 4.30% item dibatalkan

In [7]:
variance_summary = (
    df["schedule_variance_days"].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_frame("schedule_variance_days")
)

display(variance_summary.round(1))

,schedule_variance_days
count,180519.0
mean,0.6
std,1.5
min,-2.0
25%,0.0
50%,1.0
75%,1.0
90%,2.0
95%,3.0
99%,4.0
